# Bootstrap the VisDrone benchmark from GitHub

Run this same notebook in Google Colab, Kaggle, or another local Jupyter environment. It detects the host, resolves writable paths, installs only the shared stack, and runs read-only diagnostics. It never starts model training.

In [ ]:
REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_PATH = ""  # Empty selects the platform default.
REFERENCE_TYPE = "branch"  # branch, tag, or commit
REFERENCE = "main"
DRIVE_ROOT = ""  # Empty selects Drive, Kaggle working, or local artifacts.
MOUNT_GOOGLE_DRIVE = True
INSTALL_SHARED_DEPENDENCIES = True


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
IN_KAGGLE = not IN_COLAB and bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or Path("/kaggle/working").is_dir()
)
NOTEBOOK_PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
repository_default = (
    Path("/content/aerial-object-detection-benchmark")
    if IN_COLAB
    else Path("/kaggle/working/aerial-object-detection-benchmark")
    if IN_KAGGLE
    else (
        Path.cwd()
        if (Path.cwd() / "pyproject.toml").is_file()
        else Path.cwd() / "aerial-object-detection-benchmark"
    )
)
repository = Path(REPOSITORY_PATH).expanduser() if REPOSITORY_PATH else repository_default
repository = repository.resolve()
if not repository.exists():
    repository.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPOSITORY_URL, str(repository)], check=True)
elif not (repository / ".git").exists():
    raise RuntimeError(f"Existing path is not a Git checkout: {repository}")
sys.path.insert(0, str(repository))

from src.colab_setup import checkout_repository_ref
state = checkout_repository_ref(repository, REFERENCE, REFERENCE_TYPE)
print(state)


In [ ]:
from src.notebook_environment import setup_notebook_environment
from src.colab_setup import initialize_drive_directories, validate_drive_writable
from scripts.diagnostics.run_diagnostics import build_report

notebook_environment = setup_notebook_environment(
    repository,
    platform=NOTEBOOK_PLATFORM,
    artifact_root=DRIVE_ROOT or None,
    use_google_drive=MOUNT_GOOGLE_DRIVE,
    requirements_file="requirements/legacy-colab.txt",
    install_dependencies=INSTALL_SHARED_DEPENDENCIES,
)
DRIVE_ROOT = notebook_environment.artifact_root
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(paths.root)
report = build_report(repository)
print(notebook_environment.as_dict())
print(report)
if report["repository"]["dirty"]:
    raise RuntimeError("Bootstrap must finish with a clean Git checkout.")


Bootstrap complete. Continue with `00_prepare_visdrone.ipynb` at the same selected commit. Large artifacts remain under `DRIVE_ROOT`.